# Description

## Background

Model V3 suffered because inputs for various match outcomes looked very similar. A previous experiment showed that position information i.e. forward, midfield, defense and goalkeeper are irrelevant since they're similar for all team pairings. Another experiment focused on making matches with 1-1 outcome more similar found that raw goals value contributed to alot of difference. Replacing raw goals with a offense and defense value brought more 1-1 matches closer to each other. The best performance was:

1. Offense for each player being calculated as average goals per apperance if player is either midfielder or forward then having them added up for the team
2. Defense being apperances of player if they're defender of goalkeeper then added up for team

These 2 values replace goal

## Hypothesis

A 3 layer neural network architecture that's is similar to that used by Model V3 will have better performance on this new dataset

In [1]:
# Allowing notebook to import from components folder
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# Imports
import numpy as np
import numpy.typing as npt
from components.data import load_original_data_with_trimmed_y, extract_features_from_old_player_vector, collect_player_data_into_sum_concat_teams_form_from_old, split_and_normalize_dataset
from components.constants import FEW_APPEARANCES_THRESHOLD, AVG_APPEARANCES_THRESHOLD
from sklearn.model_selection import train_test_split

# Modify Data

Need to add offense defense values to the training data then normalize

In [2]:
# Function defining how to get offense defense for player
def arrange_player_details_to_offense_defense(
    raw_player: npt.NDArray
):
    """ 
    Takes in a raw player with old details and returns player with details for offense defense test i.e.
        1. age
        2. offense
        3. defense
        4. few appearances
        5. avg appearances
        6. many appearances
        7. num_tournaments
        8. average time per team
        
    With shape (1,8)
    """
    age, forward, midfielder, defender, goalkeeper, num_tournamets, apperances, goals, average_time = extract_features_from_old_player_vector(player=raw_player)
    
    # Get offense and defense
    offense = 0
    defense = 0
    if (forward == 1 or midfielder == 1):
        offense = goals / apperances if apperances > 0 else 0
    elif (defender == 1 or goalkeeper ==1):
        defense = apperances
        
    # Categorizing appearances
    few_appearances = 0
    avg_appearances = 0
    many_appearances = 0
    if (apperances < FEW_APPEARANCES_THRESHOLD):
        few_appearances = 1
    elif (FEW_APPEARANCES_THRESHOLD <= apperances < AVG_APPEARANCES_THRESHOLD):
        avg_appearances = 1
    elif (apperances >= AVG_APPEARANCES_THRESHOLD):
        many_appearances = 1
    
    return np.array([
        [age, offense, defense, few_appearances, avg_appearances, many_appearances, num_tournamets, average_time]
    ])

# Load original data
X, Y = load_original_data_with_trimmed_y()
X_offense_defense = collect_player_data_into_sum_concat_teams_form_from_old(
    Input=X, 
    num_player_features=8, 
    extract_player_vector=arrange_player_details_to_offense_defense
)
print(X_offense_defense.shape)
print(Y.shape)

(1400, 16)
(1400, 121)


In [4]:
# Split data into train, cv and test
X_train, X_cv, X_test, Y_train, Y_cv, Y_test = split_and_normalize_dataset(X=X_offense_defense, Y=Y, training_set_ratio=0.86, cv_test_set_ratio=0.5, random_state=121)

print(X_train.shape, Y_train.shape)
print(X_cv.shape, Y_cv.shape)
print(X_test.shape, Y_test.shape)

(16, 1203) (121, 1203)
(16, 98) (121, 98)
(16, 99) (121, 99)


# Train Model

Train V3 archictecure with the new data